# One-stage anchor-based detector: RetinaNet

COCO-pretrained RetinaNet ResNet50-FPN v2 (torchvision) on the adopted split: training setup, history, results, tuning experiments, the anchors and their assignment, what removing the proposal stage changes, failure analysis, and a comparison with the tuned Faster R-CNN of `02_two_stage.ipynb`.

| # | Section |
| --- | --- |
| 1 | [Setup](#1-setup) |
| 2 | [Training setup](#2-training-setup) |
| 3 | [Training history](#3-training-history) |
| 4 | [Results](#4-results) |
| 5 | [Experiments](#5-experiments) |
| 6 | [Inside the model](#6-inside-the-model) |
| 7 | [Failure analysis](#7-failure-analysis) |
| 8 | [Comparison](#8-comparison) |
| 9 | [Conclusions](#9-conclusions) |

## 1. Setup

Imports, data root from `DATA_ROOT`, the shared plot theme, the annotations of the adopted split, and the rows of `experiments/results.csv`, the source of truth for reported numbers. Run directories come from the `run_dir` column. No cell needs a checkpoint: the anchor analysis in section 6 depends only on the anchors, not on trained weights.

In [ ]:
import json
import sys
import tomllib
from pathlib import Path

import numpy as np
import polars as pl
import torch
from decouple import AutoConfig
from IPython.display import display

# make the repository root importable when the kernel starts in notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
# decouple locates .env from the caller file, which is a temp file inside a kernel
config = AutoConfig(search_path=PROJECT_ROOT)

from src.data.annotations import load_annotations, load_split_clips
from src.data.boxes import explode_size_buckets
from src.data.classes import CLASS_NAMES
from src.data.dataset import select_frames
from src.data.statistics import compute_resize_scale
from src.evaluate import METRICS_FILE, PREDICTIONS_FILE
from src.evaluation.failure_analysis import (
    compute_recall_by_size,
    count_error_categories,
    find_best_f1_threshold,
    match_detections,
    select_error_examples,
)
from src.inference.torchvision_detectors import select_device, tabulate_anchor_matches
from src.models.anchors import compute_best_anchor_iou, tabulate_anchors
from src.models.retinanet import build_retinanet
from src.training.experiment import CONFIG_FILE
from src.visualization.frames import plot_error_examples
from src.visualization.plots import plot_grouped_bars, plot_training_history
from src.visualization.style import set_plot_theme
from src.visualization.tables import compare_with_reference, load_run_histories

FAMILY = "retinanet"
BASELINE = "retinanet-baseline"

SEED = 42
set_plot_theme()
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(200)

DATA_ROOT = Path(config("DATA_ROOT"))
TUNED_FASTER_RCNN = "faster_rcnn-giou_loss"
tables = load_annotations(DATA_ROOT, load_split_clips(PROJECT_ROOT / "config" / "data.toml"))
val_images = tables.images.filter(pl.col("split") == "val")
val_objects = tables.objects.filter(pl.col("split") == "val")
val_ignored = tables.ignored.filter(pl.col("split") == "val")
results = pl.read_csv(PROJECT_ROOT / "experiments" / "results.csv")
run_dirs = {row["experiment"]: PROJECT_ROOT / row["run_dir"] for row in results.iter_rows(named=True)}
run_configs = {e: tomllib.loads((d / CONFIG_FILE).read_text()) for e, d in run_dirs.items()}
run_metrics = {e: json.loads((d / METRICS_FILE).read_text()) for e, d in run_dirs.items()}
family_results = results.filter(pl.col("experiment").str.starts_with(f"{FAMILY}-"))
family_results.select("experiment", "run_dir", "seed", "machine", "is_dirty")

Three RetinaNet runs, all on the PC from a clean working tree: the baseline and two one-factor experiments.

## 2. Training setup

The baseline protocol shared with the other families (AGENTS.md, Data and Experiments): COCO-pretrained weights, frames resized to a shorter side of 1024 px (`max_size` 1824), train frames at step 5, about 1.5 GPU hours on the PC, SGD with momentum 0.9 at a constant LR without warmup or augmentation. Values are read from the baseline run's resolved `config.toml` and `metrics.json`.

In [ ]:
baseline_config = run_configs[BASELINE]
baseline_training = run_metrics[BASELINE]["training"]
training_fields = [
    "machine", "frame_step", "epochs", "batch_size", "grad_accum", "effective_batch_size",
    "gpu_hours", "best_epoch",
]
pl.DataFrame(
    [("config", key, str(value)) for key, value in baseline_config.items() if not isinstance(value, dict)]
    + [
        ("training record", key, str(round(value, 2) if isinstance(value, float) else value))
        for key in training_fields
        for value in [baseline_training[key]]
    ],
    schema=["source", "setting", "value"],
    orient="row",
)

- 12 epochs at effective batch 4 took 1.38 GPU hours, within the 1.5 hour budget; the best monitoring epoch is 4.
- The LR is the torchvision reference 0.01 at batch 16 scaled linearly to 0.0025.

In [ ]:
pl.DataFrame(
    [
        (table, key, str(value))
        for table in ("anchors", "matching", "focal_loss", "inference")
        if table in baseline_config
        for key, value in baseline_config[table].items()
    ],
    schema=["table", "setting", "value"],
    orient="row",
)

- Anchors are the torchvision defaults: 3 sizes per octave from 32 to 812 px on P3 to P7, h / w 0.5, 1, 2, 9 per location; IoU ≥ 0.5 positive, < 0.4 negative.
- The `[focal_loss]` table is missing because the baseline run predates that option; it trained with torchvision's fixed alpha 0.25 and gamma 2.
- Inference keeps the top 1,000 candidates per FPN level, per-class NMS at IoU 0.5, and up to 500 detections.

## 3. Training history

Per-epoch monitoring on val frames at step 5 (520 frames) with the shared evaluator; `best.pt` is the epoch with the highest monitoring mAP. The 5-epoch experiments use the baseline's LR, data order, and seed, so their first epochs are directly comparable with the baseline's.

In [ ]:
family_run_dirs = {e: run_dirs[e] for e in family_results["experiment"]}
histories = load_run_histories(family_run_dirs)
fig = plot_training_history(
    histories,
    {
        "val/map": "Monitoring mAP@[.5:.95]",
        "val/map50": "Monitoring AP50",
        "val/ap_small": "Monitoring AP_small",
    },
    "RetinaNet monitoring metrics per epoch (val, step 5)",
)

- The baseline rises to 0.2676 at epoch 4, stays between 0.260 and 0.267 up to epoch 10, and drops to 0.2559 at epoch 12. Unlike Faster R-CNN and RT-DETR, which decline right after epoch 2, RetinaNet plateaus.
- Monitoring AP_small peaks earlier, at epoch 2 (0.0875).
- `anchors_eda` starts at 0.1771 and stays 0.035 to 0.062 below the baseline over its 5 epochs; `focal_loss` follows the baseline closely.

In [ ]:
fig = plot_training_history(
    histories,
    {"train/classification": "Focal classification loss", "train/bbox_regression": "Box regression loss"},
    "RetinaNet training losses per epoch (mean over the epoch's batches)",
)

- Both losses fall in every epoch for all runs.
- `anchors_eda` has a clearly higher box loss (0.30 to 0.23 against 0.23 to 0.20): with smaller anchors the regression starts further from the targets.
- The classification loss of `focal_loss` is not comparable with the others: gamma 1 down-weights easy anchors less, so the same predictions give a larger loss.

In [ ]:
histories.group_by("experiment", maintain_order=True).agg(
    pl.len().alias("epochs"),
    pl.col("epoch").get(pl.col("val/map").arg_max()).alias("best_epoch"),
    pl.col("val/map").max().round(4).alias("best_monitoring_map"),
    pl.col("val/map").last().round(4).alias("last_monitoring_map"),
    pl.col("epoch_seconds").mean().round(0).alias("seconds_per_epoch"),
    pl.col("gpu_memory_gb").max().round(2).alias("peak_gpu_memory_gb"),
)

`anchors_eda` takes 505 s per epoch against 412 s for the baseline (23% longer) at the same peak memory (9.09 GB); `focal_loss` costs the same as the baseline.

## 4. Results

Rows of `experiments/results.csv`: the shared evaluator on all 2,593 val frames with `best.pt`, `maxDets` 500, ignored regions as `iscrowd=1`. The ≤ 1920 px columns use only frames in the test size range (secondary metrics). ms/image is measured on the PC at batch 1, including the resize and NMS.

In [ ]:
RESULT_COLUMNS = [
    "experiment", "epochs", "effective_batch_size", "gpu_hours", "map", "map50", "map75",
    "ap_small", "ap_medium", "ap_large", "map_max_width_1920", "ap_small_max_width_1920",
    "ms_per_image", "public_lb",
]
family_results.select(RESULT_COLUMNS).with_columns(pl.col("gpu_hours", "ms_per_image").round(2))

- Baseline: 0.2689 mAP, 0.5116 AP50, 0.0780 AP_small at 48.6 ms/image.
- Its public LB score (0.1562) is the lowest of the four baselines and 0.113 below its val mAP, the largest gap of all families.

## 5. Experiments

Each experiment changes one factor against the baseline; the note of each row states the change and the hypothesis. RetinaNet has no seed repeat; the Faster R-CNN repeat (`02_two_stage.ipynb` §5) is the noise reference: 0.0009 full-val mAP, 0.0054 monitoring mAP per epoch on average.

In [ ]:
COMPARED_METRICS = ["map", "map50", "map75", "ap_small", "ap_medium", "ap_large", "map_max_width_1920"]
display(family_results.select("experiment", "epochs", "notes"))
compare_with_reference(family_results, BASELINE, COMPARED_METRICS).with_columns(
    pl.selectors.starts_with("delta_").round(4)
)

- `anchors_eda`: −0.0418 mAP, with AP75 −0.0776 and AP_large −0.0866; AP_small does not improve (−0.0014).
- `focal_loss` (gamma 1): −0.0018 mAP, AP_small +0.0027; within noise.

In [ ]:
pl.DataFrame(
    {
        "class_name": list(CLASS_NAMES),
        **{
            e: [round(run_metrics[e]["evaluation"]["per_class_ap"][name], 4) for name in CLASS_NAMES]
            for e in family_results["experiment"]
        },
    }
)

- `anchors_eda` loses AP on 7 of 8 classes, most on truck (−0.105) and bus (−0.098), the largest objects; only motor gains (+0.004).
- `focal_loss` changes each class by at most 0.02 (bus −0.017, truck +0.015).

In [ ]:
COMMON_EPOCHS = 5
(
    histories.filter(pl.col("epoch") <= COMMON_EPOCHS)
    .pivot(on="experiment", index="epoch", values="val/map")
    .sort("epoch")
    .select(
        "epoch",
        pl.col(BASELINE).round(4),
        *(
            (pl.col(e) - pl.col(BASELINE)).round(4).alias(f"{e} - baseline")
            for e in family_run_dirs
            if e != BASELINE
        ),
    )
)

- `focal_loss` stays within −0.0098 to +0.0015 of the baseline per epoch, inside the Faster R-CNN per-epoch seed spread (mean 0.0054, max 0.0119): gamma 1 and 2 train the same.
- `anchors_eda` is 0.035 to 0.062 below the baseline in every epoch, far outside that spread.

In [ ]:
# the tuned model: highest full-val mAP, seed repeats excluded; ties go to the earlier row
TUNED = str(
    family_results.filter(~pl.col("experiment").str.contains(r"-s\d+$"))
    .sort("map", descending=True, maintain_order=True)["experiment"]
    .first()
)
TUNED

Neither experiment improved the baseline, so `retinanet-baseline` is the tuned RetinaNet used in sections 7 and 8.

## 6. Inside the model

RetinaNet predicts a class score for each of 8 classes and a box offset for every anchor of the FPN levels P3 to P7 (strides 8 to 128), 9 anchors per location (3 sizes per octave, h / w 0.5, 1, 2). Anchors with IoU ≥ 0.5 with a GT box are positives, below 0.4 negatives, in between ignored; every GT box also keeps its best anchors. The focal loss sums over all positive and negative anchors, normalized by the number of positives. This section looks at how the anchors cover the boxes (6.1), how many positives and negatives a frame produces (6.2), and what removing the proposal stage changes (6.3).

### 6.1 Default and EDA anchors

Best IoU of every `train` box with any anchor, anchors at their feature-map grid centers (`src.models.anchors.compute_best_anchor_iou`), boxes resized like the model input; size buckets use the original box area. Thresholds 0.5 and 0.4 are RetinaNet's positive and negative IoU. `default` is the torchvision setting of the baseline (sizes 32 to 812 px), `anchors_eda` the experiment (base sizes 16 to 256 px).

In [ ]:
RETINANET_STRIDES = [8, 16, 32, 64, 128]
anchor_tables = {
    name: tabulate_anchors(
        run_configs[experiment]["anchors"]["sizes"],
        run_configs[experiment]["anchors"]["aspect_ratios"],
        RETINANET_STRIDES,
    )
    for name, experiment in [("default", BASELINE), ("anchors_eda", "retinanet-anchors_eda")]
}
display(
    pl.concat(
        table.with_columns(pl.lit(name).alias("anchor_setting")) for name, table in anchor_tables.items()
    )
    .group_by("anchor_setting", "level", "stride", maintain_order=True)
    .agg(
        pl.len().alias("anchors"),
        (pl.col("w") * pl.col("h")).sqrt().min().round(0).alias("min_sqrt_area_px"),
        (pl.col("w") * pl.col("h")).sqrt().max().round(0).alias("max_sqrt_area_px"),
    )
)
train_boxes = (
    tables.objects.filter(pl.col("split") == "train")
    .join(tables.images.select("split", "image_id", "width", "height"), on=["split", "image_id"])
    .with_columns(
        compute_resize_scale(baseline_config["min_size"], baseline_config["max_size"]).alias(
            "scale"
        )
    )
    .with_columns(pl.col("x", "y", "w", "h") * pl.col("scale"))
)
train_box_array = train_boxes.select("x", "y", "w", "h").to_numpy()
best_anchor_iou = train_boxes.select("area").with_columns(
    pl.Series(name, compute_best_anchor_iou(train_box_array, anchors, is_grid_aligned=True))
    for name, anchors in anchor_tables.items()
)
IOU_THRESHOLDS = (0.5, 0.4)
(
    pl.concat(
        [
            best_anchor_iou.with_columns(pl.lit("all").alias("size_bucket")),
            explode_size_buckets(best_anchor_iou, pl.col("area")).with_columns(
                pl.col("size_bucket").cast(pl.String)
            ),
        ]
    )
    .drop("area")
    .unpivot(index="size_bucket", variable_name="anchor_setting", value_name="best_iou")
    .group_by("anchor_setting", "size_bucket", maintain_order=True)
    .agg(
        pl.len().alias("boxes"),
        pl.col("best_iou").mean().round(3).alias("mean_best_iou"),
        *((pl.col("best_iou") < t).mean().round(3).alias(f"share_below_{t}") for t in IOU_THRESHOLDS),
    )
    .sort(
        pl.col("anchor_setting").cast(pl.Enum(list(anchor_tables))),
        pl.col("size_bucket").cast(pl.Enum(["all", "small", "medium", "large"])),
    )
)

- Default anchors: 10.3% of train boxes have best grid IoU below 0.5, 37.5% of small, 1.6% of medium, 0.2% of large boxes.
- `anchors_eda`: 1.6% below 0.5, small 5.2%, large 0.5%. These reproduce the data-analysis estimates in `docs/experiment_ideas.md`.
- The mean best IoU falls for medium (0.753 to 0.677) and large boxes (0.770 to 0.681): each anchor is now half as large relative to its stride, so an off-grid box overlaps its nearest anchor less.

### 6.2 Positive and negative anchors per frame

The anchors of every 25th `val` frame (all 7 clips) are matched to the GT boxes with RetinaNet's own matcher, exactly as in training (`src.inference.torchvision_detectors.tabulate_anchor_matches`), for both anchor settings. The matching depends only on the anchors and the boxes, so the models are built from their run configs without loading `best.pt`.

In [ ]:
ANCHOR_FRAME_STEP = 25
anchor_frames = select_frames(val_images, ANCHOR_FRAME_STEP)
anchor_frame_objects = val_objects.filter(pl.col("image_id").is_in(anchor_frames["image_id"].implode()))
device = select_device()
anchor_matches = {}
for name, experiment in [("default", BASELINE), ("anchors_eda", "retinanet-anchors_eda")]:
    detector = build_retinanet(run_configs[experiment]).to(device).eval()
    anchor_matches[name] = tabulate_anchor_matches(detector, anchor_frames, anchor_frame_objects, device)
    del detector
pl.concat(
    frames.select(
        pl.lit(name).alias("anchor_setting"),
        pl.len().alias("frames"),
        pl.col("anchors").mean().round(0).alias("anchors_per_frame"),
        pl.col("positives").mean().round(1).alias("positives_per_frame"),
        pl.col("ignored").mean().round(1).alias("ignored_per_frame"),
        pl.col("negatives").mean().round(0).alias("negatives_per_frame"),
        (pl.col("negatives") / pl.col("positives")).median().round(0).alias("median_negatives_per_positive"),
    )
    for name, (frames, _) in anchor_matches.items()
)

In [ ]:
pl.concat(
    pl.concat(
        [
            objects.with_columns(pl.lit("all").alias("size_bucket")),
            explode_size_buckets(objects, pl.col("w") * pl.col("h")).with_columns(
                pl.col("size_bucket").cast(pl.String)
            ),
        ]
    )
    .group_by("size_bucket", maintain_order=True)
    .agg(
        pl.len().alias("objects"),
        pl.col("positive_anchors").mean().round(1).alias("mean_positive_anchors"),
        pl.col("positive_anchors").median().alias("median_positive_anchors"),
        (pl.col("positive_anchors") <= 1).mean().round(3).alias("share_with_at_most_1"),
    )
    .with_columns(pl.lit(name).alias("anchor_setting"))
    for name, (_, objects) in anchor_matches.items()
).select("anchor_setting", "size_bucket", "objects", "mean_positive_anchors", "median_positive_anchors", "share_with_at_most_1")

- Every frame has about 350,000 anchors in both settings. With the default anchors a frame has 954 positive, 1,742 ignored, and 347,297 negative anchors on average: a median of 390 negatives per positive.
- With `anchors_eda` the positives fall to 276 per frame (−71%) and the median ratio rises to 1,333 negatives per positive.
- Per GT box, the default anchors give 23.3 positives on average (small 14.4, medium 24.7, large 29.8); `anchors_eda` gives 6.7 (small 6.5, medium 6.7, large 7.4). Smaller anchors reduce the positives of every size, small boxes included, although each small box now has a better best anchor (section 6.1).
- This is the measured mechanism behind the failed experiment: fewer positive training samples per object and a 3.4 times stronger imbalance, with heads pretrained for the default anchors.

### 6.3 Removing the proposal stage

Faster R-CNN filters its anchors twice: the RPN keeps 1,000 proposals per frame after NMS, and the second stage samples 512 RoIs per frame with at most 25% positives for its classifier. RetinaNet has no such filter: its classifier sees every anchor of section 6.2 at once, almost all of them easy background, which the focal loss down-weights. The tables above and below compare the two designs on the same frames and protocol; the error profiles are compared in section 7.

- **Measured.** Faster R-CNN's RPN loss samples 256 anchors per frame (at most half positive), and its classifier 512 RoIs (at most 25% positive) from 1,000 proposals. RetinaNet's focal loss sees all of about 350,000 anchors per frame at a median of 390 negatives per positive (section 6.2).
- **Consequences in this notebook.** Against the tuned Faster R-CNN, RetinaNet has lower recall (70.5% against 82.7%, small 49.3% against 66.4%, section 7.1), 0.042 lower AP_small, and 4.5 times more duplicates (section 7.2), and it is 12 ms faster per frame (section 8).
- **How the issue is usually addressed.** The focal loss down-weights the easy negatives; here its gamma made no measurable difference (section 5). Remaining options, as hypotheses: positive assignment by box statistics instead of a fixed IoU ([ATSS](https://arxiv.org/abs/1912.02424)), or anchors that cover small boxes without reducing positives per object (more sizes per octave rather than smaller sizes).

## 7. Failure analysis

The tuned RetinaNet against the tuned Faster R-CNN on all `val` frames, from their `val_predictions.parquet` (`src.evaluation.failure_analysis`). Detections are matched greedily in descending score order, class-aware, at IoU 0.5; detections at least half inside an ignored region are excluded from the counts.

### 7.1 Recall by object size

Share of GT boxes matched by any kept detection (up to 500 per frame), per COCO size bucket. This is the recall ceiling of the AP computation at IoU 0.5.

In [ ]:
ANALYSED = list(dict.fromkeys([TUNED, TUNED_FASTER_RCNN]))
val_predictions = {e: pl.read_parquet(run_dirs[e] / PREDICTIONS_FILE) for e in ANALYSED}
matches = {e: match_detections(tables, "val", val_predictions[e]) for e in ANALYSED}
recall_by_size = pl.concat(
    compute_recall_by_size(matches[e]).with_columns(pl.lit(e).alias("experiment"))
    for e in ANALYSED
)
fig = plot_grouped_bars(
    recall_by_size,
    "size_bucket",
    "recall",
    "experiment",
    ("COCO size bucket", "Recall at IoU 0.5", "Recall of all kept detections by object size (val)"),
    "{:.3f}",
)
recall_by_size.with_columns(pl.col("recall").round(3))

- With all kept detections RetinaNet matches 70.5% of val GT boxes at IoU 0.5: small 49.3%, medium 72.4%, large 95.0%. The tuned Faster R-CNN matches 82.7% (66.4%, 84.8%, 97.0%).
- Half of the small objects are never detected at any score, and the gap to Faster R-CNN is largest for small and medium boxes.

### 7.2 Error breakdown at the best-F1 threshold

The score threshold that maximizes F1 at IoU 0.5 on `val` (ignored detections excluded), then one category per detection above it: correct, duplicate (same-class GT already matched), wrong class, poor localisation (best IoU in [0.1, 0.5)), background (best IoU < 0.1); misses are GT boxes left unmatched.

In [ ]:
thresholds = pl.DataFrame(
    [{"experiment": e, **find_best_f1_threshold(matches[e])} for e in ANALYSED]
)
error_counts = pl.concat(
    count_error_categories(matches[e], thresholds.filter(pl.col("experiment") == e)["threshold"].item())
    .with_columns(pl.lit(e).alias("experiment"))
    for e in ANALYSED
)
display(thresholds.with_columns(pl.col("threshold", "precision", "recall", "f1").round(3)))
error_counts.pivot(on="experiment", index="category", values="count")

In [ ]:
fig = plot_grouped_bars(
    error_counts,
    "category",
    "count",
    "experiment",
    ("Category", "Detections or missed GT boxes", "Error breakdown at the best-F1 threshold (val)"),
    "{:.0f}",
)

- Best-F1 threshold 0.423: precision 0.710, recall 0.597, F1 0.649 (tuned Faster R-CNN 0.691).
- Misses dominate: about 42,400 GT boxes (40% of 105,240) against about 36,000 for Faster R-CNN.
- RetinaNet has fewer background detections (about 6,500 against 9,400) but more wrong-class detections (about 9,500 against 7,900) and about 4.5 times as many duplicates (about 1,100 against 240).

In [ ]:
tuned_threshold = thresholds.filter(pl.col("experiment") == TUNED)["threshold"].item()
CLASS_BY_ID = dict(enumerate(CLASS_NAMES))
(
    matches[TUNED].detections.filter(
        pl.col("score") >= tuned_threshold, pl.col("category") == "wrong_class"
    )
    .group_by("best_class_id", "class_id")
    .len()
    .sort("len", "best_class_id", "class_id", descending=[True, False, False])
    .head(8)
    .select(
        pl.col("best_class_id").replace_strict(CLASS_BY_ID).alias("gt_class"),
        pl.col("class_id").replace_strict(CLASS_BY_ID).alias("predicted_class"),
        pl.col("len").alias("detections"),
    )
)

- The largest confusions are van predicted as car (about 1,840), bicycle as motor (about 1,410), motor as person (about 1,380), car as van (about 770), and tricycle as truck (about 680).
- The pairs are those of Faster R-CNN, with more confusion between the two-wheelers.

### 7.3 Example frames per category

Tuned model at its best-F1 threshold: the four highest-scoring detections of every error category and four missed GT boxes drawn at random (seed 42), at most one per clip. Each crop is centered on the example, with all GT boxes (yellow), all kept detections (blue), and shaded ignored regions; the example has a thick outline.

In [ ]:
examples = select_error_examples(
    matches[TUNED], val_images, tuned_threshold, per_category=4, seed=SEED
)
fig = plot_error_examples(
    examples,
    val_images,
    val_objects,
    val_ignored,
    val_predictions[TUNED].filter(pl.col("score") >= tuned_threshold),
    context_scale=5.0,
    min_side=96,
)
examples.join(
    val_images.select("image_id", "clip", "frame_id"), on="image_id", maintain_order="left"
).select(
    "category",
    "clip",
    "frame_id",
    pl.col("class_id").replace_strict(CLASS_BY_ID).alias("class_name"),
    pl.col("score").round(3),
    (pl.col("w") * pl.col("h")).sqrt().round(1).alias("sqrt_area_px"),
)

- Correct: cars, at scores of about 0.96 (Faster R-CNN reaches 1.00).
- Duplicate: cars in parking areas and people in groups, where boxes overlap and NMS at IoU 0.5 keeps several of them.
- Wrong class: confident car predictions on vans.
- Poor localisation: as in Faster R-CNN, the detection fits the car while the GT box is larger.
- Background: cars without a GT box or ignored region, one at the frame edge.
- Miss: people and vehicles at night and occluded objects.

### 7.4 Failure cases

Four failure cases of the tuned RetinaNet. Counts come from the tables above; the causes are hypotheses.

1. **Small objects are missed**: recall 49.3% for small boxes with all detections. Small boxes get fewer positive anchors than large ones (14 against 30 per box, section 6.2), so they contribute fewer training samples; making the anchors smaller reduced the positives further.
2. **Duplicates** (about 1,100, 4.5 times Faster R-CNN): with no second stage to rescore and refine the boxes, several neighbouring anchors keep confident boxes on one object, and NMS at IoU 0.5 does not remove the shifted ones in dense parking areas and groups of people.
3. **Similar classes**: van as car, and the two-wheeler classes in both directions; the classifier sees each object only through its anchors' features.
4. **Label issues shared with Faster R-CNN**: loose GT boxes behind poor localisation and unlabelled vehicles behind background detections (`02_two_stage.ipynb` §7.4).

## 8. Comparison

RetinaNet against the tuned Faster R-CNN, the reference of the anchor-based families, by class and by size; the last table lists the baseline rows of all families.

In [ ]:
EXTRA_COMPARED = ["faster_rcnn-baseline"]

In [ ]:
COMPARED = list(dict.fromkeys([*ANALYSED, *EXTRA_COMPARED]))
per_class_ap = pl.DataFrame(
    {
        "class_name": list(CLASS_NAMES),
        **{
            e: [round(run_metrics[e]["evaluation"]["per_class_ap"][name], 4) for name in CLASS_NAMES]
            for e in COMPARED
        },
    }
)
per_class_ap

- RetinaNet is below the tuned Faster R-CNN on 7 of 8 classes; only bus is higher (0.408 against 0.384).
- The largest gaps are bicycle (−0.078), tricycle (−0.068), and motor (−0.050), the small and mid-sized two- and three-wheelers.

In [ ]:
compare_with_reference(
    results.filter(pl.col("experiment").is_in(COMPARED)),
    TUNED_FASTER_RCNN,
    ["map", "map50", "map75", "ap_small", "ap_medium", "ap_large", "ms_per_image"],
).with_columns(pl.selectors.float().round(4))

Against the tuned Faster R-CNN, RetinaNet loses 0.0377 mAP: AP_small −0.0423, AP_medium −0.0577, AP_large −0.0346. It is 12.1 ms per frame faster (48.6 against 60.8 ms).

In [ ]:
results.filter(pl.col("experiment").str.ends_with("-baseline")).select(
    "experiment", "epochs", "gpu_hours", "map", "map50", "ap_small", "ap_medium", "ap_large",
    "ms_per_image", "public_lb",
).with_columns(pl.col("gpu_hours", "ms_per_image").round(2))

- RetinaNet's val mAP (0.2689) is close to YOLO11s (0.2671), but its public LB score is much lower (0.1562 against 0.2097).
- RT-DETRv2 has the best baseline (0.3209 val, 0.2563 LB), YOLO11s is the fastest (13.7 ms/image).

## 9. Conclusions

**Results**

- **Baseline.** COCO-pretrained RetinaNet ResNet50-FPN v2 reaches 0.2689 mAP, 0.5116 AP50, and 0.0780 AP_small on full val at 48.6 ms/image after 1.38 GPU hours (public LB 0.1562). It plateaus from epoch 4 to 10 instead of declining right after an early peak (§3).
- **Tuning.** Anchors one octave smaller (`anchors_eda`) lose 0.0418 mAP without any AP_small gain; focal loss gamma 1 instead of 2 changes nothing measurable (−0.0018, per epoch within seed noise). The baseline stays the tuned RetinaNet (§5).
- **Anchors.** The EDA anchors cover small boxes much better (best IoU < 0.5 for 5.2% instead of 37.5%) but cut the positive anchors per object from 23.3 to 6.7, small boxes included (14.4 to 6.5), and raise the median imbalance from 390 to 1,333 negatives per positive (§6.1, §6.2).
- **Without a proposal stage.** RetinaNet trains on all of about 350,000 anchors per frame. Against the tuned Faster R-CNN it has lower recall (70.5% against 82.7%, small 49.3% against 66.4%), 0.0377 lower mAP, 4.5 times more duplicates, and is 12 ms faster (§6.3, §7, §8).

**Hypotheses (not tested here)**

- For anchor-based one-stage training, the number of positive anchors per object matters more than the best-anchor IoU: an anchor change should keep or raise positives per object, e.g. more sizes per octave or an adaptive assignment such as ATSS.
- The missing second stage explains the extra duplicates and the lower recall on small objects; RetinaNet is not a strong ensemble member next to Faster R-CNN and RT-DETR.